In [1]:
import torch as th
from datasets import load_dataset
import random
from transformers import AutoTokenizer, AutoModelForCausalLM, RobertaTokenizer, RobertaForSequenceClassification, pipeline, BitsAndBytesConfig
import lqr_utils_seq as lqr
from functools import partial
import pickle
# from steering import LQRSteering
# from PIDsteering import PIDSteering
from datasets import load_dataset
import random
import time
from tqdm import tqdm

/home/jskifstad/labcode/ctrlgpt/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
device = th.device("cuda" if th.cuda.is_available() else "cpu")

model_name = "google/gemma-2-2b"
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,          # or load_in_8bit=True
    bnb_4bit_compute_dtype=th.float16,
    bnb_4bit_quant_type="nf4",  # best for LLMs
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=quant_config, dtype=th.float32, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.eos_token_id

Loading checkpoint shards: 100%|██████████| 3/3 [00:35<00:00, 11.79s/it]


In [9]:
test = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
not_pos = sorted([test[i]['text'] for i in range(len(test)) if test[i]['text'].strip()], key=len)

count = 0
totalPPL = 0

BATCH_SZ = 10
batch_ppls = []

In [ ]:
# for ind in tqdm(range(0, 10, BATCH_SZ)):
#     if not not_pos[ind]:
#         continue

#     end_ind = min(ind+BATCH_SZ, len(not_pos))
#     encodings = tokenizer(not_pos[ind:end_ind], return_tensors="pt", padding=True, truncation=True).to(device)
#     # encodings = tokenizer("".join(test["text"]), return_tensors="pt")

#     # print("text:", test['text'])

#     max_length = min(model.config.max_position_embeddings, 4096)
#     print(f"MAX LENGTH: {max_length}")
#     stride = 1
#     seq_len = encodings.input_ids.size(1)

#     input_ids = encodings["input_ids"]
#     attention_mask = encodings["attention_mask"]
#     sentence_length = attention_mask.sum(-1)
#     ppls = th.zeros(attention_mask.shape[0], device=device)
#     totals = th.zeros_like(ppls)

#     nll_sum = 0.0
#     n_tokens = 0
#     prev_end_loc = 0
#     nlls = []
#     print(input_ids.shape)
#     # for begin_loc in tqdm(range(0, seq_len-1, stride)):
#     for ctx_len in range(1, sentence_length.max()-1):
#         mask = ctx_len < sentence_length
#         curr_ids = input_ids[mask][:,:ctx_len]
#         curr_attn_mask = attention_mask[mask][:,:ctx_len]

#         # print(f"CHUNK: {chunk.shape}")
#         with th.no_grad():
#             outputs = model(input_ids=curr_ids,
#                             attention_mask=curr_attn_mask,
#                             use_cache=False)
#             logits = outputs.logits
#             loss = th.nn.functional.cross_entropy(
#                 logits[:, -1],
#                 input_ids[mask][:,ctx_len].reshape(-1).to(device),
#                 reduction="none",
#             )
#         # print(loss)
#         ppls[mask] += loss
#         totals[mask] += 1


#     # print(ppl)
#     print(ppls)
#     print(totals)
#     print(f"batch ppl: {th.exp(ppls / totals)}")
#     batch_ppls.append(th.mean(th.exp(ppls / totals)))

# print(f"the grand total (average) PPL: {batch_ppls}")

  0%|          | 0/1 [00:00<?, ?it/s]

MAX LENGTH: 4096
torch.Size([10, 258])


100%|██████████| 1/1 [01:24<00:00, 84.00s/it]

tensor([ 41.1302, 686.4111, 731.1547,  41.1302, 111.5895, 833.9743, 764.0157,
         88.1170, 754.7141, 754.5568], device='cuda:0')
tensor([  7., 196., 208.,   7.,  19., 156., 215.,  15., 210., 256.],
       device='cuda:0')
batch ppl: tensor([356.2877,  33.1850,  33.6215, 356.2877, 355.3607, 209.7653,  34.9375,
        355.8358,  36.3748,  19.0580], device='cuda:0')
the grand total (average) PPL: [tensor(179.0714, device='cuda:0'), tensor(179.0714, device='cuda:0')]


In [11]:
import torch.nn.functional as F


total_nll = 0.0
total_tokens = 0

# for ind in tqdm(range(0, len(not_pos), BATCH_SZ)):
for ind in tqdm(range(12, 22, 4)):
    if not not_pos[ind]:
        continue

    end_ind = min(ind + BATCH_SZ, len(not_pos))

    encodings = tokenizer(
        not_pos[ind:end_ind],
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=model.config.max_position_embeddings,
    ).to(device)

    input_ids = encodings["input_ids"]
    attention_mask = encodings["attention_mask"]

    with th.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        # Shift for causal LM
        shift_logits = outputs.logits[:, :-1, :]
        shift_labels = input_ids[:, 1:]
        shift_mask = attention_mask[:, 1:]

        # Token-level NLL (sum, not mean)
        # print("logits shift", shift_logits.shape)
        # print("reshape:", shift_logits.reshape(-1, shift_logits.size(-1)))
        # print("labels shift", shift_labels.shape)
        loss = F.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=tokenizer.eos_token_id,
            reduction="sum",
        )

        # Count valid tokens
        n_tokens = shift_mask.sum()

    total_nll += loss
    total_tokens += n_tokens

# Final perplexity (HF definition)
ppl = th.exp(total_nll / total_tokens)
print(f"Final PPL: {ppl}")


100%|██████████| 3/3 [00:01<00:00,  2.99it/s]

Final PPL: 13817.423828125


In [12]:

from steering import LQRSteering
PKL_FILENAME = "../../pickle_jar/"
nontox_filename = "gemma-2-2b_nontox"
with open(PKL_FILENAME+nontox_filename+".pkl", "rb") as f:
    loaded_tensors = pickle.load(f)

# Access tensors
X = loaded_tensors["X"]
A = loaded_tensors["A"]
print(f"X shape: {X.shape}")
print(f"A shape: {A.shape}")

tox_filename = "gemma-2-2b_tox"
with open(PKL_FILENAME+tox_filename+".pkl", "rb") as f:
    loaded_tensors = pickle.load(f)

    # Access tensors
X_tox = loaded_tensors["X"]

X_contr = X - X_tox

for l in [0.5, 1, 1.5, 2, 2.5]:
    steer = LQRSteering(model, tokenizer, q=0.1,r=1,qf=0.1, A=A, contrastive_vecs=X_contr)

    ppl = steer.compute_ppl(not_pos, lmbda=l, BATCH_SZ=5)
    print(f"lambda: {l}, steered_ppl: {ppl}")

X shape: torch.Size([27, 2304])
A shape: torch.Size([26, 2304, 2304])
betas= [tensor(6.2188, device='cuda:0'), tensor(5.1782, device='cuda:0'), tensor(5.6771, device='cuda:0'), tensor(6.1186, device='cuda:0'), tensor(6.6644, device='cuda:0'), tensor(7.7627, device='cuda:0'), tensor(8.5758, device='cuda:0'), tensor(8.9758, device='cuda:0'), tensor(10.9800, device='cuda:0'), tensor(11.5984, device='cuda:0'), tensor(12.1872, device='cuda:0'), tensor(12.9571, device='cuda:0'), tensor(14.1002, device='cuda:0'), tensor(14.1254, device='cuda:0'), tensor(17.0471, device='cuda:0'), tensor(17.3194, device='cuda:0'), tensor(20.1737, device='cuda:0'), tensor(22.2050, device='cuda:0'), tensor(23.5913, device='cuda:0'), tensor(26.0760, device='cuda:0'), tensor(28.4656, device='cuda:0'), tensor(31.3595, device='cuda:0'), tensor(36.7221, device='cuda:0'), tensor(41.3045, device='cuda:0'), tensor(44.2871, device='cuda:0'), tensor(51.3904, device='cuda:0'), tensor(60.9568, device='cuda:0')]


100%|██████████| 2/2 [00:00<00:00,  2.52it/s]


lambda: 0.5, steered_ppl: 18.364246368408203
betas= [tensor(12.4375, device='cuda:0'), tensor(10.3563, device='cuda:0'), tensor(11.3542, device='cuda:0'), tensor(12.2373, device='cuda:0'), tensor(13.3287, device='cuda:0'), tensor(15.5254, device='cuda:0'), tensor(17.1516, device='cuda:0'), tensor(17.9516, device='cuda:0'), tensor(21.9601, device='cuda:0'), tensor(23.1967, device='cuda:0'), tensor(24.3745, device='cuda:0'), tensor(25.9143, device='cuda:0'), tensor(28.2005, device='cuda:0'), tensor(28.2507, device='cuda:0'), tensor(34.0942, device='cuda:0'), tensor(34.6387, device='cuda:0'), tensor(40.3473, device='cuda:0'), tensor(44.4100, device='cuda:0'), tensor(47.1826, device='cuda:0'), tensor(52.1521, device='cuda:0'), tensor(56.9312, device='cuda:0'), tensor(62.7190, device='cuda:0'), tensor(73.4442, device='cuda:0'), tensor(82.6091, device='cuda:0'), tensor(88.5741, device='cuda:0'), tensor(102.7809, device='cuda:0'), tensor(121.9136, device='cuda:0')]


100%|██████████| 2/2 [00:00<00:00,  2.54it/s]


lambda: 1, steered_ppl: 18.364246368408203
betas= [tensor(18.6563, device='cuda:0'), tensor(15.5345, device='cuda:0'), tensor(17.0313, device='cuda:0'), tensor(18.3559, device='cuda:0'), tensor(19.9931, device='cuda:0'), tensor(23.2881, device='cuda:0'), tensor(25.7274, device='cuda:0'), tensor(26.9274, device='cuda:0'), tensor(32.9401, device='cuda:0'), tensor(34.7951, device='cuda:0'), tensor(36.5617, device='cuda:0'), tensor(38.8714, device='cuda:0'), tensor(42.3007, device='cuda:0'), tensor(42.3761, device='cuda:0'), tensor(51.1413, device='cuda:0'), tensor(51.9581, device='cuda:0'), tensor(60.5210, device='cuda:0'), tensor(66.6151, device='cuda:0'), tensor(70.7738, device='cuda:0'), tensor(78.2281, device='cuda:0'), tensor(85.3968, device='cuda:0'), tensor(94.0785, device='cuda:0'), tensor(110.1663, device='cuda:0'), tensor(123.9136, device='cuda:0'), tensor(132.8612, device='cuda:0'), tensor(154.1713, device='cuda:0'), tensor(182.8704, device='cuda:0')]


...

lambda: 1.5, steered_ppl: 18.364246368408203
betas= [tensor(24.8750, device='cuda:0'), tensor(20.7127, device='cuda:0'), tensor(22.7084, device='cuda:0'), tensor(24.4746, device='cuda:0'), tensor(26.6575, device='cuda:0'), tensor(31.0508, device='cuda:0'), tensor(34.3032, device='cuda:0'), tensor(35.9031, device='cuda:0'), tensor(43.9202, device='cuda:0'), tensor(46.3934, device='cuda:0'), tensor(48.7490, device='cuda:0'), tensor(51.8285, device='cuda:0'), tensor(56.4010, device='cuda:0'), tensor(56.5014, device='cuda:0'), tensor(68.1884, device='cuda:0'), tensor(69.2775, device='cuda:0'), tensor(80.6946, device='cuda:0'), tensor(88.8201, device='cuda:0'), tensor(94.3651, device='cuda:0'), tensor(104.3041, device='cuda:0'), tensor(113.8624, device='cuda:0'), tensor(125.4380, device='cuda:0'), tensor(146.8883, device='cuda:0'), tensor(165.2182, device='cuda:0'), tensor(177.1482, device='cuda:0'), tensor(205.5618, device='cuda:0'), tensor(243.8272, device='cuda:0')]


100%|██████████| 2/2 [00:00<00:00,  2.55it/s]


lambda: 2, steered_ppl: 18.364246368408203
betas= [tensor(31.0938, device='cuda:0'), tensor(25.8909, device='cuda:0'), tensor(28.3855, device='cuda:0'), tensor(30.5932, device='cuda:0'), tensor(33.3218, device='cuda:0'), tensor(38.8135, device='cuda:0'), tensor(42.8790, device='cuda:0'), tensor(44.8789, device='cuda:0'), tensor(54.9002, device='cuda:0'), tensor(57.9918, device='cuda:0'), tensor(60.9362, device='cuda:0'), tensor(64.7856, device='cuda:0'), tensor(70.5012, device='cuda:0'), tensor(70.6268, device='cuda:0'), tensor(85.2355, device='cuda:0'), tensor(86.5968, device='cuda:0'), tensor(100.8683, device='cuda:0'), tensor(111.0251, device='cuda:0'), tensor(117.9564, device='cuda:0'), tensor(130.3801, device='cuda:0'), tensor(142.3280, device='cuda:0'), tensor(156.7975, device='cuda:0'), tensor(183.6104, device='cuda:0'), tensor(206.5227, device='cuda:0'), tensor(221.4353, device='cuda:0'), tensor(256.9522, device='cuda:0'), tensor(304.7841, device='cuda:0')]


100%|██████████| 2/2 [00:00<00:00,  2.59it/s]

lambda: 2.5, steered_ppl: 18.364246368408203


In [15]:

from PIDsteering import PIDSteering
PKL_FILENAME = "../../pickle_jar/"
nontox_filename = "gemma-2-2b_nontox"
with open(PKL_FILENAME+nontox_filename+".pkl", "rb") as f:
    loaded_tensors = pickle.load(f)

# Access tensors
X = loaded_tensors["X"]
A = loaded_tensors["A"]
print(f"X shape: {X.shape}")
print(f"A shape: {A.shape}")

tox_filename = "gemma-2-2b_tox"
with open(PKL_FILENAME+tox_filename+".pkl", "rb") as f:
    loaded_tensors = pickle.load(f)

    # Access tensors
X_tox = loaded_tensors["X"]

X_contr = X - X_tox

for l in [0.5, 1, 1.5, 2, 2.5]:
    steer = PIDSteering(model, tokenizer, kp=0.5,ki=0.01,kd=0.01, A=A, contrastive_vecs=X_contr)

    ppl = steer.compute_ppl(not_pos, lmbda=l, BATCH_SZ=5)
    print(f"lambda: {l}, steered_ppl: {ppl}")

X shape: torch.Size([27, 2304])
A shape: torch.Size([26, 2304, 2304])


100%|██████████| 2/2 [00:00<00:00,  2.55it/s]


lambda: 0.5, steered_ppl: 18.364246368408203


100%|██████████| 2/2 [00:00<00:00,  2.54it/s]


lambda: 1, steered_ppl: 18.364246368408203


...

lambda: 1.5, steered_ppl: 18.364246368408203


100%|██████████| 2/2 [00:00<00:00,  2.59it/s]


lambda: 2, steered_ppl: 18.364246368408203


100%|██████████| 2/2 [00:00<00:00,  2.59it/s]

lambda: 2.5, steered_ppl: 18.364246368408203


In [ ]:
import pandas as pd
csv_path = "../data/wikipedia_sentences.csv"
df = pd.read_csv(csv_path)
all_sentences = df["text"].values.tolist()
all_sentences = [s.replace("<s> ", "") for s in all_sentences]
# print(all_sentences[0])

prompts = None

tokenizer.padding_side = "right"
truncation = True
max_generation_length = 50
max_context_length = 128
# max_context_length = model.config.max_position_embeddings

BATCH_SZ = 1


nll_sum = th.zeros(1)
total = th.zeros(1)

for i in tqdm(range(0, len(all_sentences), BATCH_SZ)):
# for i in range(21, 24, BATCH_SZ):
    end_ind = min(i+BATCH_SZ, len(all_sentences)-1)
    sentences = all_sentences[i:end_ind]
    # print(f"string length: {len(sentences[0])}")
    # First tokenize sentence tokens since we will need them anyways
    tok_s = tokenizer(
        text=sentences,
        return_tensors="pt",
        truncation=truncation,
        padding=truncation,
        max_length=max_generation_length,
        add_special_tokens=(
            prompts is None
        ),  # if there is a prompt, it already contains BOS token
    ).to(device)
    tokenizer.padding_side = (
        "left"  # go back to original padding (to not messup things)
    )

    if prompts is not None:
        # Now we tokenize the prompts
        side = tokenizer.truncation_side
        # The sentence is the direct continuation of the prompt, so we truncate the prompt by the left
        tokenizer.truncation_side = "left"
        tok_p = tokenizer(
            text=prompts,
            return_tensors="pt",
            truncation=truncation,
            padding=True,
            add_special_tokens=True,
            max_length=max_context_length,
        ).to(device)
        tokenizer.truncation_side = side
        # Concatenate prompt tokens with sentence tokens.
        # This is the only way to know exactly at which token the prompt ends and the sentence starts.
        # (Note that tokenizer(prompt+continuation) != cat([tokenizer(prompt), tokenizer(continuation)]))
        tok_all = {k: th.cat([tok_p[k], tok_s[k]], -1) for k in tok_p.keys()}
        # This tells us where prompts end and sentences start, so we can slice them later on.
        offset = tok_p["input_ids"].shape[-1]
    else:
        tok_all = tok_s
        offset = 1  # skips the BOS token

    input_ids = tok_all["input_ids"]
    # print(f"shape; {input_ids.shape}")
    attention_mask = tok_all["attention_mask"]
    # This is the number of tokens in each continuation. We will generate this amount of tokens, one by one.
    attention_mask_sum = tok_s["attention_mask"].sum(-1)
    # Buffer to keep track of ppls
    # ppls = th.zeros(attention_mask.shape[0], device=device, dtype=th.float32)
    # totals = th.zeros_like(ppls)

    # nll = 0
    # Now we iterate a cursor over all continuations at the same time (they all start at the same position and end up at different positions, marked by attention_mask_sum)
    # print(f"sum: {attention_mask_sum.max()}")
    for ctx_len in tqdm(range(1, attention_mask_sum.max() - 1)):
        # We can stop computing perplexity for those continuations that have reached an end.
        mask = ctx_len < attention_mask_sum
        # print(f"mask shape: {mask.shape}")
        # Pick all tokens since prompt beginning to prompt + current cursor position
        _input_ids = input_ids[mask][:, : (offset + ctx_len)]
        _attention_mask = attention_mask[mask][:, : (offset + ctx_len)]
        logits = model(input_ids=_input_ids, attention_mask=_attention_mask, use_cache=False).logits
        # Compute perplexity for last token (note that indexing at offset + ctx_len gives us the token id right after :(offset + ctx_len))
        loss = th.nn.functional.cross_entropy(
            logits[:, -1],
            input_ids[mask][:, (offset + ctx_len)].reshape(-1),
            reduction="none",
        )
        # del logits
        # ppls[mask] += loss
        # totals[mask] += 1
        nll_sum += loss.item()
        total += 1

    # print("batch ppl:", th.exp(ppls / totals))

print(f"sequential ppl: {th.exp(nll_sum/ total)}")


  0%|          | 0/100 [00:00<?, ?it/s]

string length: 16424
shape; torch.Size([1, 50])
sum: 50


  1%|          | 1/100 [00:18<30:00, 18.19s/it]

string length: 14861
shape; torch.Size([1, 50])
sum: 50


  2%|▏         | 2/100 [00:36<29:30, 18.06s/it]

...

string length: 470
shape; torch.Size([1, 50])
sum: 50


  4%|▍         | 4/100 [01:12<28:46, 17.99s/it]

string length: 271
shape; torch.Size([1, 50])
sum: 50


  4%|▍         | 4/100 [01:22<32:52, 20.54s/it]


KeyboardInterrupt: 